# 05. LSTM

**LSTM** (Hochreiter & Schmidhuber, 1997) — mô hình chính:
$$i_\tau, f_\tau, o_\tau = \sigma(W_{i,f,o} x_\tau + U_{i,f,o} h_{\tau-1} + b_{i,f,o}),\quad \tilde c_\tau = \tanh(W_c x_\tau + U_c h_{\tau-1} + b_c)$$
$$c_\tau = f_\tau \odot c_{\tau-1} + i_\tau \odot \tilde c_\tau,\qquad h_\tau = o_\tau \odot \tanh(c_\tau)$$

**Kiến trúc chung** (`src/models/recurrent.py`): chuỗi L × F → khối hồi quy (1–2 lớp, hidden ∈ {32, 64, 128}) → trạng thái ẩn
cuối → Dropout 0,2 → (ghép mưa tương lai ở FUT) → Linear(H). Huấn luyện: Huber có mặt nạ (× trọng số ở E5, + λ·phạt dự báo thiếu
ở E6), Adam 1e-3, batch 64, dừng sớm theo val (kiên nhẫn 20).

**Quy trình (giống hệt cho mọi mô hình, mã ở `src/pipeline.py`)**
1. Dò siêu tham số trên tập biến F1 (E2), hàm mất mát cơ bản, bài toán t+1, CV 3 fold; tiêu chí: NSE CV (trung bình 3 fold).
2. Thực nghiệm trên CV với siêu tham số đã chọn: E1–E4 (vai trò Hủa Na), thực nghiệm phụ (F4, F5, EMA), E5 (trọng số
   $w=(1+Q/Q_{95,train})^\gamma$, γ ∈ {1, 2}, chặn trên ∈ {không, 20}, chọn theo CV), E6 / E56 (phạt dự báo thiếu — chỉ LSTM, GRU),
   FUT (mưa thực đo t+1, cận trên).
3. Huấn luyện lại trên 2019–2024 (số epoch / số cây = trung vị vòng tốt nhất ở CV), dự báo tập test **khoá** 01/2025 – 08/2026.
4. Bài toán mở rộng t+1 … t+10 cho E1, E2, E5 (cùng siêu tham số).

In [ ]:
# ── Google Colab: gắn Google Drive, vào thư mục dự án, cài đúng phiên bản thư viện. Chạy trên máy thì tự bỏ qua. ──
import os, sys, glob, subprocess
DU_AN = '/content/drive/MyDrive/T_DAThS_Ninh'   # ← thư mục dự án trên Google Drive (notebook 01, mục 0 tạo ra)
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.isdir(os.path.join(DU_AN, 'src')):          # không thấy → tự tìm trong My Drive (sâu tối đa 3 cấp)
        hit = [h for k in range(1, 4) for h in glob.glob('/content/drive/MyDrive/' + '*/' * k + 'src/results_store.py')]
        assert hit, ('Chưa có dự án trên Google Drive: mở notebook 01_PhanTich_XuLy_DuLieu và chạy 2 ô của mục 0 '
                     '(lấy code từ GitHub, giải nén data/kết quả), hoặc sửa biến DU_AN ở đầu ô này')
        DU_AN = os.path.dirname(os.path.dirname(hit[0]))
    os.chdir(DU_AN)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements_colab.txt'], check=True)

import json, warnings
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / 'src' / 'results_store.py').exists():
    assert ROOT != ROOT.parent, 'Không tìm thấy thư mục dự án (thư mục chứa src/): mở notebook từ trong dự án'
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_colwidth', 80)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .3})
from src.utils import load_config, results_dir, dataset_meta
cfg = load_config()
print('Thư mục dự án:', ROOT, '| kết quả lưu ở:', cfg['paths']['results'])
print('Thiết bị:', f'GPU {torch.cuda.get_device_name(0)}' if torch.cuda.is_available() else
      'CPU (trên Colab: Runtime → Change runtime type → GPU để chạy nhanh hơn)')
from src import pipeline
from src.training.train import DISPLAY

## LSTM — dò siêu tham số

In [ ]:
tab, best = pipeline.tune(cfg, 'lstm', verbose=False)
print(DISPLAY['lstm'], '→ siêu tham số chọn:', best)
display(tab.head(10).round(3))

### LSTM: thực nghiệm trên CV (t+1)

In [ ]:
cvt = pipeline.cv_experiments(cfg, 'lstm', verbose=False)
display(cvt.round(3)); print('Cấu hình:', pipeline.load_conf(cfg, 'lstm'))

### LSTM: huấn luyện cuối 2019–2024 và test khoá (t+1)

In [ ]:
met = pipeline.final_test(cfg, 'lstm', verbose=False)
cols = ['NSE', 'KGE', 'r2', 'RMSE', 'MAE', 'PBIAS', 'NSE_Q95', 'RMSE_Q95', 'Peak_RMSE', 'Peak_Bias', 'Peak_Timing', 'n_iter']
display(met.groupby('exp', sort=False)[cols].agg(['mean', 'std']).round(3))

### LSTM: thí nghiệm A — cửa sổ chung L = 3

In [ ]:
print(DISPLAY['lstm'], '→', pipeline.run_common_L(cfg, 'lstm', 3, verbose=False))

### LSTM: bài toán mở rộng t+1 … t+10

In [ ]:
cv10, met10 = pipeline.extended(cfg, 'lstm', verbose=False)
display(cv10.round(3)); display(met10.groupby('exp', sort=False)[['NSE', 'KGE', 'RMSE', 'PBIAS', 'NSE_Q95', 'Peak_Bias']].mean().round(3))